# Evolución de las GANs: Vanilla GAN → WGAN → WGAN-GP

**Autor:** Enrique Oliva (214205)  
**Curso:** Inteligencia Artificial Generativa 2025  
**Tema:** Análisis Comparativo de la Evolución de las Redes Generativas Adversariales

---

## Objetivo

Este notebook presenta un análisis empírico de la evolución de las Generative Adversarial Networks (GANs) desde GANs "vanilla" hasta WGAN-GPs, mostrando cómo cada arquitectura sucesiva soluciona problemas fundamentales de su predecesora:

### Vanilla GAN (Goodfellow et al., 2014)
**Problemas identificados:**
1. **Training instability**: Pérdidas que oscilan erráticamente sin indicar calidad
2. **Mode collapse**: El generador produce poca variedad
3. **Vanishing gradients**: Cuando D es óptimo, G no recibe gradientes útiles
4. **Métrica no correlacionada**: BCE loss no indica calidad de generación

### WGAN (Arjovsky et al., 2017)
**Soluciones aportadas:**
- Wasserstein distance como métrica correlacionada con calidad
- No saturación de gradientes (incluso con D óptimo)
- Training más estable

**Problemas introducidos:**
- Weight clipping es una restricción tosca
- Capacidad reducida del modelo (pesos forzados a extremos [-c, c])
- Gradientes pueden explotar o desaparecer

### WGAN-GP (Gulrajani et al., 2017)
**Soluciones definitivas:**
- Gradient penalty reemplaza weight clipping (restricción suave vs. dura)
- Distribución de pesos natural (sin confinamiento artificial)
- Gradientes estables
- Menor sensibilidad a hiperparámetros

---

## Enfoque Experimental

Implementaremos y entrenaremos las tres arquitecturas en condiciones controladas para demostrar empíricamente:
1. Los problemas concretos de Vanilla GAN
2. Cómo WGAN los soluciona (y qué problemas introduce)
3. Cómo WGAN-GP soluciona los problemas de WGAN

### Referencias
- [Vanilla GAN Paper (Goodfellow et al., 2014)](https://arxiv.org/abs/1406.2661)
- [WGAN Paper (Arjovsky et al., 2017)](https://arxiv.org/abs/1701.07875)
- [WGAN-GP Paper (Gulrajani et al., 2017)](https://arxiv.org/abs/1704.00028)

## 1. Configuración e Importaciones

In [ ]:
# Para correr en Google Colabs (descomentar):

# !pip install wandb
# import wandb
# wandb.login()
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_OUTPUT_DIR = "/content/drive/MyDrive/wgan_outputs"
# DATA_DIR = "/content/drive/MyDrive/wgan_data"

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import torchvision
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from datetime import datetime
import wandb

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        pass

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Hiperparámetros y Configuración General

Definimos todos los hiperparámetros en un solo lugar para facilitar la experimentación.

In [ ]:
BATCH_SIZE = 64
IMAGE_SIZE = 32
NUM_CHANNELS = 3
LATENT_DIM = 128
GENERATOR_FEATURES = 128
CRITIC_FEATURES = 128

CRITIC_ITERATIONS = 5
NUM_EPOCHS = 1

VANILLA_LEARNING_RATE = 2e-4
VANILLA_BETAS = (0.5, 0.999)

WGAN_LEARNING_RATE = 5e-5
WGAN_CLIP_VALUE = 0.01

WGANGP_LEARNING_RATE = 1e-4
WGANGP_LAMBDA = 10
WGANGP_BETAS = (0.5, 0.9)

Configuración general

In [ ]:
BASE_OUTPUT_DIR = "./outputs"
EXPERIMENT_NAME = "wgan_comparison"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

RESULTS_DIR = f"{BASE_OUTPUT_DIR}/{EXPERIMENT_NAME}_{TIMESTAMP}"
MODELS_DIR = f"{RESULTS_DIR}/models"
FIGURES_DIR = f"{RESULTS_DIR}/figures"
SAMPLES_DIR = f"{RESULTS_DIR}/samples"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

DATA_DIR = "./data"

WANDB_PROJECT = "Obligatorio de IA Generativa 2025"
WANDB_ENTITY = "eo214205-ort"

WANDB_GROUP = datetime.now().strftime("%d/%m/%y - %H:%M")

USE_WANDB = False

print("Configuración cargada correctamente.")
print(f"Épocas de entrenamiento: {NUM_EPOCHS}")
print(f"wandb online logging: {'ACTIVADO' if USE_WANDB else 'DESACTIVADO'}")
print(f"wandb project: {WANDB_PROJECT}")
print(f"wandb group: {WANDB_GROUP}")
print(f"\nDirectorios de salida:")
print(f"  - Modelos:  {MODELS_DIR}")
print(f"  - Figuras:  {FIGURES_DIR}")
print(f"  - Muestras: {SAMPLES_DIR}")
print(f"  - Datos:    {DATA_DIR}")

## 3. Carga de Datos (CIFAR-10)

Utilizamos CIFAR-10 (32x32 RGB) como dataset de prueba, usado en los 3 papers

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, 
    train=True,
    download=True, 
    transform=transform
)

dataloader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,
    shuffle=True, 
    num_workers=0,
    drop_last=True
)

print(f"Dataset: CIFAR-10")
print(f"Tamaño del dataset: {len(train_dataset)} imágenes")
print(f"Batches por época: {len(dataloader)}")
print(f"Tamaño de imagen: {IMAGE_SIZE}x{IMAGE_SIZE}x{NUM_CHANNELS}")

In [ ]:
def show_images(images, title="Imágenes", nrow=8):
    plt.figure(figsize=(10, 10))
    plt.axis("off")
    plt.title(title, fontsize=14)
    plt.imshow(np.transpose(
        vutils.make_grid(images[:64], padding=2, normalize=True, nrow=nrow).cpu(),
        (1, 2, 0)
    ))
    plt.show()

real_batch = next(iter(dataloader))[0]
show_images(real_batch, "Muestras Reales de CIFAR-10")

## 4. Arquitecturas de Redes Neuronales

Implementamos las arquitecturas para las tres variantes de GANs.

### 4.1 Arquitecturas para Vanilla GAN

En Vanilla GAN, tanto el Generador como el Discriminador usan **sigmoid** en sus salidas y se entrenan con **Binary Cross-Entropy Loss**.

**Diferencias clave con WGAN/WGAN-GP:**
- El Discriminador regresa probabilidades entre [0,1] (usa sigmoid)
- Se usa BatchNorm en ambas redes
- Función de pérdida: BCE en vez de Wasserstein distance

In [ ]:
class VanillaDiscriminator(nn.Module):
    def __init__(self, num_channels, num_features):
        super(VanillaDiscriminator, self).__init__()
        
        self.main = nn.Sequential(
            nn.Conv2d(num_channels, num_features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features, num_features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(num_features * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features * 2, num_features * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(num_features * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.BatchNorm2d)):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
    
    def forward(self, x):
        return self.main(x).view(-1)

test_disc = VanillaDiscriminator(NUM_CHANNELS, CRITIC_FEATURES).to(device)
test_img = torch.randn(1, NUM_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
test_out = test_disc(test_img)
print(f"Vanilla Discriminator: {NUM_CHANNELS}x{IMAGE_SIZE}x{IMAGE_SIZE} -> probabilidad ({test_out.shape})")
print(f"Rango de salida: [{test_out.item():.4f}] (debe estar en [0,1])")
del test_disc, test_img, test_out

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim, num_features, num_channels):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, num_features * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(num_features * 4),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(num_features * 4, num_features * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(num_features * 2),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(num_features * 2, num_features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(num_features),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(num_features, num_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.BatchNorm2d)):
                nn.init.normal_(m.weight.data, 0.0, 0.02) 
    
    def forward(self, z):
        return self.main(z)

test_gen = Generator(LATENT_DIM, GENERATOR_FEATURES, NUM_CHANNELS).to(device)
test_z = torch.randn(1, LATENT_DIM, 1, 1, device=device)
test_out = test_gen(test_z)
print(f"Generador: {LATENT_DIM} -> {test_out.shape[1]}x{test_out.shape[2]}x{test_out.shape[3]}")
del test_gen, test_z, test_out

### 4.2 Generador (Compartido por las 3 variantes)

El Generador es idéntico para Vanilla GAN, WGAN y WGAN-GP. Transforma un vector latente z ∈ ℝ^128 en una imagen de 32x32x3.

**Características clave:**
- Salida con **Tanh** (produce valores en [-1,1], matching la normalización de CIFAR-10)
- Usa BatchNorm y ReLU en capas intermedias
- Arquitectura basada en convoluciones transpuestas (upsampling progresivo)

###  4.3 Critic para WGAN/WGAN-GP

El Critic evalúa qué tan "real" es una imagen. A diferencia del Vanilla Discriminator, el Critic **no** tiene sigmoid al final (regresa un escalar sin bounds). 

**Diferencias cruciales con Vanilla Discriminator:**
- **Sin sigmoid**: Regresa valores sin restricción (no probabilidades)
- **Sin BatchNorm**: BatchNorm crea dependencias entre muestras de un batch, normalizando estadísticas compartidas
  - Esto interfiere con el **gradient penalty** en WGAN-GP, que requiere evaluar cada muestra independientemente
  - En WGAN con clipping, BatchNorm puede interactuar mal con la operación de recorte de pesos

In [ ]:
class Critic(nn.Module):
    def __init__(self, num_channels, num_features):
        super(Critic, self).__init__()
        
        self.main = nn.Sequential(
            nn.Conv2d(num_channels, num_features, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features, num_features * 2, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features * 2, num_features * 4, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(num_features * 4, 1, 4, 1, 0, bias=False)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
    
    def forward(self, x):
        return self.main(x).view(-1)

test_critic = Critic(NUM_CHANNELS, CRITIC_FEATURES).to(device)
test_img = torch.randn(1, NUM_CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=device)
test_out = test_critic(test_img)
print(f"Critic: {NUM_CHANNELS}x{IMAGE_SIZE}x{IMAGE_SIZE} -> escalar ({test_out.shape})")
del test_critic, test_img, test_out

## 5. Funciones de Utilidad

Implementamos funciones para monitorear el entrenamiento y capturar métricas importantes.

### 5.1 Gradient Penalty (para WGAN-GP)

In [ ]:
def compute_gradient_penalty(critic, real_samples, fake_samples, device):
    batch_size = real_samples.size(0)
    
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    alpha = alpha.expand_as(real_samples)
    
    interpolated = alpha * real_samples + (1 - alpha) * fake_samples
    interpolated.requires_grad_(True)
    
    critic_interpolated = critic(interpolated)
    
    gradients = autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,
        retain_graph=True
    )[0]
    
    gradients = gradients.view(batch_size, -1)
    gradient_norm = gradients.norm(2, dim=1)
    gradient_penalty = ((gradient_norm - 1) ** 2).mean()
    
    return gradient_penalty

### 5.2 Logging de Gradientes por Capa (Experimento 3)

In [ ]:
def get_gradient_norms(model):
    gradient_norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            gradient_norms[name] = param.grad.data.norm(2).item()
    return gradient_norms

def get_weight_statistics(model):
    weight_stats = {}
    for name, param in model.named_parameters():
        if 'weight' in name:
            weight_stats[name] = {
                'mean': param.data.mean().item(),
                'std': param.data.std().item(),
                'min': param.data.min().item(),
                'max': param.data.max().item(),
                'values': param.data.cpu().numpy().flatten()
            }
    return weight_stats

### 5.3 Clase para Almacenar Métricas

In [ ]:
class TrainingMetrics:
    def __init__(self):
        self.critic_losses = []
        self.generator_losses = []
        self.wasserstein_distances = []
        self.gradient_norms_history = []
        self.weight_stats_history = []
        self.training_time = 0
        self.generated_samples = []
        self.iterations = []
    
    def log_losses(self, iteration, c_loss, g_loss, w_distance):
        self.iterations.append(iteration)
        self.critic_losses.append(c_loss)
        self.generator_losses.append(g_loss)
        self.wasserstein_distances.append(w_distance)
    
    def log_gradients(self, critic_grads, gen_grads):
        self.gradient_norms_history.append({
            'critic': critic_grads,
            'generator': gen_grads
        })
    
    def log_weights(self, critic_weights, gen_weights):
        self.weight_stats_history.append({
            'critic': critic_weights,
            'generator': gen_weights
        })
    
    def save_samples(self, samples):
        self.generated_samples.append(samples.detach().cpu())

## 6. Entrenamiento WGAN (Weight Clipping)

WGAN utiliza weight clipping para forzar la restricción de Lipschitz. Después de cada actualización del Critic, los pesos se recortan al rango [-c, c].

**Problema conocido**: El weight clipping puede causar:
- Capacidad reducida del modelo (pesos forzados a los extremos)
- Gradientes que explotan o desaparecen

## 6. Funciones de Entrenamiento

Implementamos las funciones de entrenamiento para las tres variantes de GANs.

### 6.1 Entrenamiento Vanilla GAN

Vanilla GAN usa **Binary Cross-Entropy Loss** para entrenar D y G:
- **Discriminador**: Maximiza log(D(x)) + log(1 - D(G(z)))
- **Generador**: Minimiza log(1 - D(G(z))) o equivalentemente maximiza log(D(G(z)))

**Problemas esperados:**
- Las pérdidas no correlacionan con calidad de generación
- Posible vanishing gradients cuando D es muy bueno
- Training instability

In [ ]:
def train_vanilla_gan(dataloader, num_epochs, learning_rate=VANILLA_LEARNING_RATE,
                      betas=VANILLA_BETAS, log_interval=100, use_wandb=True):

    run_name = f"Vanilla GAN"

    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            group=WANDB_GROUP,
            name=run_name,
            config={
                "modelo": "Vanilla GAN",
                "learning_rate": learning_rate,
                "epocas": num_epochs,
                "batch_size": BATCH_SIZE,
                "latent_dim": LATENT_DIM,
                "optimizador": "Adam",
                "adam_betas": betas,
                "dataset": "CIFAR-10",
                "tamaño_imagen": IMAGE_SIZE,
                "loss_function": "BCELoss"
            }
        )

    print(f"Entrenando Vanilla GAN: lr={learning_rate}, épocas={num_epochs}")

    generator = Generator(LATENT_DIM, GENERATOR_FEATURES, NUM_CHANNELS).to(device)
    discriminator = VanillaDiscriminator(NUM_CHANNELS, CRITIC_FEATURES).to(device)

    criterion = nn.BCELoss()

    optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=betas)
    optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=betas)

    metrics = TrainingMetrics()
    fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)

    real_label = 1.0
    fake_label = 0.0

    start_time = time.time()
    iteration = 0

    for epoch in range(num_epochs):
        epoch_d_loss = 0
        epoch_g_loss = 0
        num_batches = 0

        for i, (real_images, _) in enumerate(dataloader):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            optimizer_D.zero_grad()

            labels_real = torch.full((batch_size,), real_label, dtype=torch.float, device=device)
            output_real = discriminator(real_images)
            d_loss_real = criterion(output_real, labels_real)
            d_loss_real.backward()

            z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_images = generator(z)
            labels_fake = torch.full((batch_size,), fake_label, dtype=torch.float, device=device)
            output_fake = discriminator(fake_images.detach())
            d_loss_fake = criterion(output_fake, labels_fake)
            d_loss_fake.backward()

            d_loss = d_loss_real + d_loss_fake
            optimizer_D.step()

            optimizer_G.zero_grad()
            labels_for_g = torch.full((batch_size,), real_label, dtype=torch.float, device=device)
            output_fake_for_g = discriminator(fake_images)
            g_loss = criterion(output_fake_for_g, labels_for_g)
            g_loss.backward()
            optimizer_G.step()

            d_x = output_real.mean().item()
            d_g_z = output_fake.mean().item()

            if iteration % log_interval == 0:
                metrics.log_losses(iteration, d_loss.item(), g_loss.item(), d_x - d_g_z)

                disc_grads = get_gradient_norms(discriminator)
                gen_grads = get_gradient_norms(generator)
                metrics.log_gradients(disc_grads, gen_grads)

                if use_wandb:
                    wandb.log({
                        "iteracion": iteration,
                        "perdida_discriminador": d_loss.item(),
                        "perdida_generador": g_loss.item(),
                        "D(x)": d_x,
                        "D(G(z))": d_g_z,
                        "epoca": epoch + 1,
                        "norma_gradiente_discriminador_total": sum(disc_grads.values()),
                        "norma_gradiente_generador_total": sum(gen_grads.values())
                    })

            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()
            num_batches += 1
            iteration += 1

        with torch.no_grad():
            fake_samples = generator(fixed_noise)
            metrics.save_samples(fake_samples)

            if use_wandb:
                sample_grid = vutils.make_grid(fake_samples[:16], padding=2, normalize=True, nrow=4)
                wandb.log({
                    "muestras_generadas": wandb.Image(sample_grid, caption=f"Época {epoch+1}"),
                    "epoca": epoch + 1
                })

        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            disc_weights = get_weight_statistics(discriminator)
            gen_weights = get_weight_statistics(generator)
            metrics.log_weights(disc_weights, gen_weights)

            if use_wandb:
                first_layer_name = list(disc_weights.keys())[0]
                wandb.log({
                    "peso_discriminador_min": disc_weights[first_layer_name]['min'],
                    "peso_discriminador_max": disc_weights[first_layer_name]['max'],
                    "peso_discriminador_std": disc_weights[first_layer_name]['std'],
                    "epoca": epoch + 1
                })
        
        print(f"\rÉpoca {epoch+1}/{num_epochs}: D_loss={epoch_d_loss/num_batches:.4f}, G_loss={epoch_g_loss/num_batches:.4f}, D(x)={d_x:.4f}, D(G(z))={d_g_z:.4f}", end='', flush=True)

    print()
    metrics.training_time = time.time() - start_time

    if use_wandb:
        wandb.log({"tiempo_entrenamiento_minutos": metrics.training_time / 60})
        wandb.finish()

    print(f"Vanilla GAN completado: {metrics.training_time/60:.2f} min")

    return generator, discriminator, metrics

### 6.2 Entrenamiento WGAN (Weight Clipping)

WGAN utiliza weight clipping para forzar la restricción de Lipschitz. Después de cada actualización del Critic, los pesos se recortan al rango [-c, c].

#### El Crítico (ya no Discriminador)

**Cambio importante:** En WGAN, el Discriminador se llama "Crítico" (Critic) porque ya no clasifica (0 o 1), sino que da un puntaje real.

#### La Restricción de Lipschitz

Para que la matemática funcione, el Crítico debe ser una función 1-Lipschitz. Esto significa que su gradiente no puede ser mayor que 1.

**¿Qué es Lipschitz?**
Una función no puede cambiar más rápido que k veces la distancia de entrada.

**¿Cómo se ejecuta en WGAN?**
Con weight clipping (recorte de pesos):

Después de cada actualización del Crítico

```python
for param in critic.parameters():
    param.data.clamp_(-c, c) 
```

Esto fuerza todos los pesos a estar en el rango [-c, c].

**Mejoras sobre Vanilla GAN:**
- Wasserstein distance como métrica significativa (correlacionada con calidad)
- No sufre vanishing gradients (incluso con Critic óptimo)

**Problemas introducidos:**
- Capacidad reducida del modelo (pesos forzados a los extremos)
- Gradientes pueden explotar o desaparecer debido al clipping abrupto

In [ ]:
def train_wgan(dataloader, num_epochs, clip_value=WGAN_CLIP_VALUE, 
               learning_rate=WGAN_LEARNING_RATE, critic_iterations=CRITIC_ITERATIONS,
               log_interval=100, use_wandb=True):
    
    run_name = f"WGAN (c={clip_value})"
    
    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            group=WANDB_GROUP,
            name=run_name,
            config={
                "modelo": "WGAN",
                "learning_rate": learning_rate,
                "epocas": num_epochs,
                "batch_size": BATCH_SIZE,
                "latent_dim": LATENT_DIM,
                "optimizador": "RMSprop",
                "clip_value": clip_value,
                "critic_iterations": critic_iterations,
                "dataset": "CIFAR-10",
                "tamaño_imagen": IMAGE_SIZE
            }
        )
    
    print(f"Entrenando WGAN: lr={learning_rate}, c={clip_value}, épocas={num_epochs}")
    
    generator = Generator(LATENT_DIM, GENERATOR_FEATURES, NUM_CHANNELS).to(device)
    critic = Critic(NUM_CHANNELS, CRITIC_FEATURES).to(device)
    
    optimizer_G = optim.RMSprop(generator.parameters(), lr=learning_rate)
    optimizer_C = optim.RMSprop(critic.parameters(), lr=learning_rate)
    
    metrics = TrainingMetrics()
    fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
    
    start_time = time.time()
    iteration = 0
    
    for epoch in range(num_epochs):
        epoch_c_loss = 0
        epoch_g_loss = 0
        num_batches = 0
        
        for i, (real_images, _) in enumerate(dataloader):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            for _ in range(critic_iterations):
                optimizer_C.zero_grad()
                
                z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
                fake_images = generator(z).detach()
                
                critic_real = critic(real_images).mean()
                critic_fake = critic(fake_images).mean()
                
                c_loss = -(critic_real - critic_fake)
                c_loss.backward()
                optimizer_C.step()
                
                for p in critic.parameters():
                    p.data.clamp_(-clip_value, clip_value)
            
            optimizer_G.zero_grad()
            
            z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_images = generator(z)
            
            g_loss = -critic(fake_images).mean()
            g_loss.backward()
            optimizer_G.step()
            
            wasserstein_distance = critic_real.item() - critic_fake.item()
            
            if iteration % log_interval == 0:
                metrics.log_losses(iteration, c_loss.item(), g_loss.item(), wasserstein_distance)
                
                critic_grads = get_gradient_norms(critic)
                gen_grads = get_gradient_norms(generator)
                metrics.log_gradients(critic_grads, gen_grads)
                
                if use_wandb:
                    wandb.log({
                        "iteracion": iteration,
                        "perdida_critic": c_loss.item(),
                        "perdida_generador": g_loss.item(),
                        "wasserstein_distance": wasserstein_distance,
                        "epoca": epoch + 1,
                        "norma_gradiente_critic_total": sum(critic_grads.values()),
                        "norma_gradiente_generador_total": sum(gen_grads.values())
                    })
            
            epoch_c_loss += c_loss.item()
            epoch_g_loss += g_loss.item()
            num_batches += 1
            iteration += 1
        
        with torch.no_grad():
            fake_samples = generator(fixed_noise)
            metrics.save_samples(fake_samples)
            
            if use_wandb:
                sample_grid = vutils.make_grid(fake_samples[:16], padding=2, normalize=True, nrow=4)
                wandb.log({
                    "muestras_generadas": wandb.Image(sample_grid, caption=f"Época {epoch+1}"),
                    "epoca": epoch + 1
                })
        
        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            critic_weights = get_weight_statistics(critic)
            gen_weights = get_weight_statistics(generator)
            metrics.log_weights(critic_weights, gen_weights)
            
            if use_wandb:
                first_layer_name = list(critic_weights.keys())[0]
                wandb.log({
                    "peso_critic_min": critic_weights[first_layer_name]['min'],
                    "peso_critic_max": critic_weights[first_layer_name]['max'],
                    "peso_critic_std": critic_weights[first_layer_name]['std'],
                    "epoca": epoch + 1
                })
        
        print(f"\rÉpoca {epoch+1}/{num_epochs}: C_loss={epoch_c_loss/num_batches:.4f}, G_loss={epoch_g_loss/num_batches:.4f}, W-dist={wasserstein_distance:.4f}", end='', flush=True)
    
    print()
    metrics.training_time = time.time() - start_time
    
    if use_wandb:
        wandb.log({"tiempo_entrenamiento_minutos": metrics.training_time / 60})
        wandb.finish()
    
    print(f"WGAN completado: {metrics.training_time/60:.2f} min")
    
    return generator, critic, metrics

### 6.3 Entrenamiento WGAN-GP (Gradient Penalty)

WGAN-GP reemplaza el weight clipping con una penalización por gradiente. En lugar de forzar los pesos, penaliza cuando la norma del gradiente se desvía de 1.

**Mejoras sobre WGAN:**
- No restringe artificialmente los pesos
- Mejor utilización de la capacidad del modelo
- Gradientes más estables y predecibles
- Menor sensibilidad a hiperparámetros

In [ ]:
def train_wgangp(dataloader, num_epochs, lambda_gp=WGANGP_LAMBDA,
                 learning_rate=WGANGP_LEARNING_RATE, betas=WGANGP_BETAS,
                 critic_iterations=CRITIC_ITERATIONS, log_interval=100, use_wandb=True):
    
    run_name = f"WGAN-GP (λ={lambda_gp})"
    
    if use_wandb:
        wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            group=WANDB_GROUP,
            name=run_name,
            config={
                "modelo": "WGAN-GP",
                "learning_rate": learning_rate,
                "epocas": num_epochs,
                "batch_size": BATCH_SIZE,
                "latent_dim": LATENT_DIM,
                "optimizador": "Adam",
                "adam_betas": betas,
                "lambda_gp": lambda_gp,
                "critic_iterations": critic_iterations,
                "dataset": "CIFAR-10",
                "tamaño_imagen": IMAGE_SIZE
            }
        )
    
    print(f"Entrenando WGAN-GP: lr={learning_rate}, λ={lambda_gp}, épocas={num_epochs}")
    
    generator = Generator(LATENT_DIM, GENERATOR_FEATURES, NUM_CHANNELS).to(device)
    critic = Critic(NUM_CHANNELS, CRITIC_FEATURES).to(device)
    
    optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=betas)
    optimizer_C = optim.Adam(critic.parameters(), lr=learning_rate, betas=betas)
    
    metrics = TrainingMetrics()
    fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=device)
    
    start_time = time.time()
    iteration = 0
    
    for epoch in range(num_epochs):
        epoch_c_loss = 0
        epoch_g_loss = 0
        num_batches = 0
        
        for i, (real_images, _) in enumerate(dataloader):
            real_images = real_images.to(device)
            batch_size = real_images.size(0)
            
            for _ in range(critic_iterations):
                optimizer_C.zero_grad()
                
                z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
                fake_images = generator(z)
                
                critic_real = critic(real_images).mean()
                critic_fake = critic(fake_images).mean()
                
                gp = compute_gradient_penalty(critic, real_images, fake_images, device)
                
                c_loss = -(critic_real - critic_fake) + lambda_gp * gp
                c_loss.backward()
                optimizer_C.step()
            
            optimizer_G.zero_grad()
            
            z = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_images = generator(z)
            
            g_loss = -critic(fake_images).mean()
            g_loss.backward()
            optimizer_G.step()
            
            wasserstein_distance = critic_real.item() - critic_fake.item()
            
            if iteration % log_interval == 0:
                metrics.log_losses(iteration, c_loss.item(), g_loss.item(), wasserstein_distance)
                
                critic_grads = get_gradient_norms(critic)
                gen_grads = get_gradient_norms(generator)
                metrics.log_gradients(critic_grads, gen_grads)
                
                if use_wandb:
                    wandb.log({
                        "iteracion": iteration,
                        "perdida_critic": c_loss.item(),
                        "perdida_generador": g_loss.item(),
                        "wasserstein_distance": wasserstein_distance,
                        "gradient_penalty": gp.item(),
                        "epoca": epoch + 1,
                        "norma_gradiente_critic_total": sum(critic_grads.values()),
                        "norma_gradiente_generador_total": sum(gen_grads.values())
                    })
            
            epoch_c_loss += c_loss.item()
            epoch_g_loss += g_loss.item()
            num_batches += 1
            iteration += 1
        
        with torch.no_grad():
            fake_samples = generator(fixed_noise)
            metrics.save_samples(fake_samples)
            
            if use_wandb:
                sample_grid = vutils.make_grid(fake_samples[:16], padding=2, normalize=True, nrow=4)
                wandb.log({
                    "muestras_generadas": wandb.Image(sample_grid, caption=f"Época {epoch+1}"),
                    "epoca": epoch + 1
                })
        
        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            critic_weights = get_weight_statistics(critic)
            gen_weights = get_weight_statistics(generator)
            metrics.log_weights(critic_weights, gen_weights)
            
            if use_wandb:
                first_layer_name = list(critic_weights.keys())[0]
                wandb.log({
                    "peso_critic_min": critic_weights[first_layer_name]['min'],
                    "peso_critic_max": critic_weights[first_layer_name]['max'],
                    "peso_critic_std": critic_weights[first_layer_name]['std'],
                    "epoca": epoch + 1
                })
        
        print(f"\rÉpoca {epoch+1}/{num_epochs}: C_loss={epoch_c_loss/num_batches:.4f}, G_loss={epoch_g_loss/num_batches:.4f}, W-dist={wasserstein_distance:.4f}", end='', flush=True)
    
    print()
    metrics.training_time = time.time() - start_time
    
    if use_wandb:
        wandb.log({"tiempo_entrenamiento_minutos": metrics.training_time / 60})
        wandb.finish()
    
    print(f"WGAN-GP completado: {metrics.training_time/60:.2f} min")
    
    return generator, critic, metrics

## 7. Funciones de Visualización y Análisis

Implementamos funciones para visualizar y comparar los resultados de las tres variantes.

In [ ]:
def plot_training_curves_all3(vanilla_metrics, wgan_metrics, wgangp_metrics):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(vanilla_metrics.iterations, vanilla_metrics.critic_losses, 
                    label='Vanilla GAN', alpha=0.7, color='green')
    axes[0, 0].plot(wgan_metrics.iterations, wgan_metrics.critic_losses, 
                    label='WGAN', alpha=0.7, color='blue')
    axes[0, 0].plot(wgangp_metrics.iterations, wgangp_metrics.critic_losses, 
                    label='WGAN-GP', alpha=0.7, color='red')
    axes[0, 0].set_title('Pérdida del Discriminador/Critic', fontsize=12)
    axes[0, 0].set_xlabel('Iteración')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(vanilla_metrics.iterations, vanilla_metrics.generator_losses, 
                    label='Vanilla GAN', alpha=0.7, color='green')
    axes[0, 1].plot(wgan_metrics.iterations, wgan_metrics.generator_losses, 
                    label='WGAN', alpha=0.7, color='blue')
    axes[0, 1].plot(wgangp_metrics.iterations, wgangp_metrics.generator_losses, 
                    label='WGAN-GP', alpha=0.7, color='red')
    axes[0, 1].set_title('Pérdida del Generador', fontsize=12)
    axes[0, 1].set_xlabel('Iteración')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(vanilla_metrics.iterations, vanilla_metrics.wasserstein_distances, 
                    label='Vanilla (D(x) - D(G(z)))', alpha=0.7, color='green', linestyle='--')
    axes[1, 0].plot(wgan_metrics.iterations, wgan_metrics.wasserstein_distances, 
                    label='WGAN', alpha=0.7, color='blue')
    axes[1, 0].plot(wgangp_metrics.iterations, wgangp_metrics.wasserstein_distances, 
                    label='WGAN-GP', alpha=0.7, color='red')
    axes[1, 0].set_title('Distancia de Wasserstein (estimada)', fontsize=12)
    axes[1, 0].set_xlabel('Iteración')
    axes[1, 0].set_ylabel('W-distance')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    training_times = [
        vanilla_metrics.training_time/60, 
        wgan_metrics.training_time/60, 
        wgangp_metrics.training_time/60
    ]
    bars = axes[1, 1].bar(['Vanilla GAN', 'WGAN', 'WGAN-GP'], training_times, 
                          color=['green', 'blue', 'red'], alpha=0.7)
    axes[1, 1].set_title('Tiempo de Entrenamiento', fontsize=12)
    axes[1, 1].set_ylabel('Minutos')
    for bar, time_val in zip(bars, training_times):
        axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{time_val:.1f}', ha='center', fontsize=11)
    
    plt.suptitle('Comparación de Curvas de Entrenamiento: 3 Variantes de GAN', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/training_curves_all3.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def compare_samples_all3(vanilla_metrics, wgan_metrics, wgangp_metrics):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    vanilla_samples = vanilla_metrics.generated_samples[-1][:64]
    vanilla_grid = vutils.make_grid(vanilla_samples, padding=2, normalize=True, nrow=8)
    axes[0].imshow(np.transpose(vanilla_grid.numpy(), (1, 2, 0)))
    axes[0].set_title('Vanilla GAN (BCE Loss)', fontsize=12)
    axes[0].axis('off')
    
    wgan_samples = wgan_metrics.generated_samples[-1][:64]
    wgan_grid = vutils.make_grid(wgan_samples, padding=2, normalize=True, nrow=8)
    axes[1].imshow(np.transpose(wgan_grid.numpy(), (1, 2, 0)))
    axes[1].set_title('WGAN (Weight Clipping)', fontsize=12)
    axes[1].axis('off')
    
    wgangp_samples = wgangp_metrics.generated_samples[-1][:64]
    wgangp_grid = vutils.make_grid(wgangp_samples, padding=2, normalize=True, nrow=8)
    axes[2].imshow(np.transpose(wgangp_grid.numpy(), (1, 2, 0)))
    axes[2].set_title('WGAN-GP (Gradient Penalty)', fontsize=12)
    axes[2].axis('off')
    
    plt.suptitle('Comparación de Muestras Finales: 3 Variantes de GAN', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/samples_all3.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_gradient_norms_all3(vanilla_metrics, wgan_metrics, wgangp_metrics):
    if not vanilla_metrics.gradient_norms_history or not wgan_metrics.gradient_norms_history or not wgangp_metrics.gradient_norms_history:
        print("No hay datos de gradientes para visualizar.")
        return
    
    vanilla_critic_norms = []
    wgan_critic_norms = []
    wgangp_critic_norms = []
    
    for grad_dict in vanilla_metrics.gradient_norms_history:
        total_norm = sum(grad_dict['critic'].values())
        vanilla_critic_norms.append(total_norm)
    
    for grad_dict in wgan_metrics.gradient_norms_history:
        total_norm = sum(grad_dict['critic'].values())
        wgan_critic_norms.append(total_norm)
    
    for grad_dict in wgangp_metrics.gradient_norms_history:
        total_norm = sum(grad_dict['critic'].values())
        wgangp_critic_norms.append(total_norm)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    ax.plot(vanilla_critic_norms, label='Vanilla GAN', alpha=0.7, color='green', linewidth=2)
    ax.plot(wgan_critic_norms, label='WGAN', alpha=0.7, color='blue', linewidth=2)
    ax.plot(wgangp_critic_norms, label='WGAN-GP', alpha=0.7, color='red', linewidth=2)
    ax.set_title('Norma Total de Gradientes del Discriminador/Critic', fontsize=14)
    ax.set_xlabel('Checkpoint', fontsize=12)
    ax.set_ylabel('Norma L2', fontsize=12)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Análisis de Estabilidad de Gradientes: 3 Variantes de GAN', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/gradient_norms_all3.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nEstadísticas de Gradientes:")
    print("="*70)
    print(f"Vanilla GAN - Media: {np.mean(vanilla_critic_norms):.2f}, Varianza: {np.var(vanilla_critic_norms):.2f}")
    print(f"WGAN        - Media: {np.mean(wgan_critic_norms):.2f}, Varianza: {np.var(wgan_critic_norms):.2f}")
    print(f"WGAN-GP     - Media: {np.mean(wgangp_critic_norms):.2f}, Varianza: {np.var(wgangp_critic_norms):.2f}")

In [ ]:
def plot_gradient_norms(wgan_metrics, wgangp_metrics):
    if not wgan_metrics.gradient_norms_history or not wgangp_metrics.gradient_norms_history:
        print("No hay datos de gradientes para visualizar.")
        return
    
    wgan_critic_norms = []
    wgangp_critic_norms = []
    
    for grad_dict in wgan_metrics.gradient_norms_history:
        total_norm = sum(grad_dict['critic'].values())
        wgan_critic_norms.append(total_norm)
    
    for grad_dict in wgangp_metrics.gradient_norms_history:
        total_norm = sum(grad_dict['critic'].values())
        wgangp_critic_norms.append(total_norm)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(wgan_critic_norms, label='WGAN', alpha=0.7, color='blue')
    axes[0].plot(wgangp_critic_norms, label='WGAN-GP', alpha=0.7, color='red')
    axes[0].set_title('Norma Total de Gradientes del Critic', fontsize=12)
    axes[0].set_xlabel('Checkpoint')
    axes[0].set_ylabel('Norma L2')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    wgan_last_grads = wgan_metrics.gradient_norms_history[-1]['critic']
    wgangp_last_grads = wgangp_metrics.gradient_norms_history[-1]['critic']
    
    layer_names = list(wgan_last_grads.keys())
    short_names = [name.split('.')[-2] + '.' + name.split('.')[-1] for name in layer_names]
    
    x = np.arange(len(layer_names))
    width = 0.35
    
    wgan_values = [wgan_last_grads[name] for name in layer_names]
    wgangp_values = [wgangp_last_grads[name] for name in layer_names]
    
    axes[1].bar(x - width/2, wgan_values, width, label='WGAN', color='blue', alpha=0.7)
    axes[1].bar(x + width/2, wgangp_values, width, label='WGAN-GP', color='red', alpha=0.7)
    axes[1].set_title('Gradientes por Capa (última iteración)', fontsize=12)
    axes[1].set_xlabel('Capa')
    axes[1].set_ylabel('Norma L2')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(short_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Análisis de Comportamiento de Gradientes (Experimento 3)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/gradient_norms_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def plot_weight_distributions(wgan_metrics, wgangp_metrics):
    if not wgan_metrics.weight_stats_history or not wgangp_metrics.weight_stats_history:
        print("No hay datos de pesos para visualizar.")
        return
    
    wgan_weights = wgan_metrics.weight_stats_history[-1]['critic']
    wgangp_weights = wgangp_metrics.weight_stats_history[-1]['critic']
    
    layer_names = [name for name in wgan_weights.keys() if 'weight' in name]
    num_layers = min(4, len(layer_names))
    
    fig, axes = plt.subplots(num_layers, 2, figsize=(14, 3*num_layers))
    if num_layers == 1:
        axes = axes.reshape(1, -1)
    
    for i, layer_name in enumerate(layer_names[:num_layers]):
        wgan_vals = wgan_weights[layer_name]['values']
        wgangp_vals = wgangp_weights[layer_name]['values']
        
        axes[i, 0].hist(wgan_vals, bins=50, alpha=0.7, color='blue', density=True)
        axes[i, 0].axvline(x=-WGAN_CLIP_VALUE, color='red', linestyle='--', 
                          label=f'Clip bounds (±{WGAN_CLIP_VALUE})')
        axes[i, 0].axvline(x=WGAN_CLIP_VALUE, color='red', linestyle='--')
        axes[i, 0].set_title(f'WGAN - {layer_name.split(".")[-2]}', fontsize=10)
        axes[i, 0].set_xlabel('Valor del peso')
        axes[i, 0].set_ylabel('Densidad')
        axes[i, 0].legend(fontsize=8)
        
        axes[i, 1].hist(wgangp_vals, bins=50, alpha=0.7, color='red', density=True)
        axes[i, 1].set_title(f'WGAN-GP - {layer_name.split(".")[-2]}', fontsize=10)
        axes[i, 1].set_xlabel('Valor del peso')
        axes[i, 1].set_ylabel('Densidad')
    
    plt.suptitle('Distribución de Pesos del Critic (Experimento 5)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/weight_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nEstadísticas de Pesos del Critic:")
    print("="*60)
    for layer_name in layer_names[:num_layers]:
        print(f"\n{layer_name}:")
        print(f"  WGAN:    min={wgan_weights[layer_name]['min']:.6f}, "
              f"max={wgan_weights[layer_name]['max']:.6f}, "
              f"std={wgan_weights[layer_name]['std']:.6f}")
        print(f"  WGAN-GP: min={wgangp_weights[layer_name]['min']:.6f}, "
              f"max={wgangp_weights[layer_name]['max']:.6f}, "
              f"std={wgangp_weights[layer_name]['std']:.6f}")

In [ ]:
def plot_sample_evolution(metrics, title, num_samples=5):
    if not metrics.generated_samples:
        print("No hay muestras generadas para visualizar.")
        return
    
    total_epochs = len(metrics.generated_samples)
    indices = np.linspace(0, total_epochs-1, num_samples, dtype=int)
    
    fig, axes = plt.subplots(1, num_samples, figsize=(3*num_samples, 3))
    
    for idx, epoch_idx in enumerate(indices):
        samples = metrics.generated_samples[epoch_idx][:16]
        grid = vutils.make_grid(samples, padding=2, normalize=True, nrow=4)
        axes[idx].imshow(np.transpose(grid.numpy(), (1, 2, 0)))
        axes[idx].set_title(f'Época {epoch_idx+1}', fontsize=10)
        axes[idx].axis('off')
    
    plt.suptitle(f'Evolución de Muestras Generadas - {title}', 
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/sample_evolution_{title.lower().replace(" ", "_")}.png', 
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def compare_final_samples(wgan_metrics, wgangp_metrics):
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    wgan_samples = wgan_metrics.generated_samples[-1][:64]
    wgan_grid = vutils.make_grid(wgan_samples, padding=2, normalize=True, nrow=8)
    axes[0].imshow(np.transpose(wgan_grid.numpy(), (1, 2, 0)))
    axes[0].set_title('WGAN (Weight Clipping)', fontsize=12)
    axes[0].axis('off')
    
    wgangp_samples = wgangp_metrics.generated_samples[-1][:64]
    wgangp_grid = vutils.make_grid(wgangp_samples, padding=2, normalize=True, nrow=8)
    axes[1].imshow(np.transpose(wgangp_grid.numpy(), (1, 2, 0)))
    axes[1].set_title('WGAN-GP (Gradient Penalty)', fontsize=12)
    axes[1].axis('off')
    
    plt.suptitle('Comparación de Muestras Finales', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/final_samples_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

---

# EXPERIMENTOS

---

## Experimento 1: Estableciendo Baselines - Las 3 Variantes

Entrenamos las tres arquitecturas con sus hiperparámetros recomendados para establecer baselines:
- **Vanilla GAN**: Adam, lr=2e-4, betas=(0.5, 0.999), BCE Loss
- **WGAN**: RMSprop, lr=5e-5, weight clipping c=0.01
- **WGAN-GP**: Adam, lr=1e-4, betas=(0.5, 0.9), gradient penalty λ=10

In [ ]:
print("="*70)
print("EXPERIMENTO 1: Estableciendo Baselines - Las 3 Variantes")
print("="*70)
print("\n[1/3] Entrenando Vanilla GAN...")
print("-"*70)

vanilla_gen, vanilla_disc, vanilla_metrics = train_vanilla_gan(
    dataloader, 
    num_epochs=NUM_EPOCHS,
    learning_rate=VANILLA_LEARNING_RATE,
    betas=VANILLA_BETAS,
    use_wandb=USE_WANDB
)

print("\n[2/3] Entrenando WGAN...")
print("-"*70)

wgan_gen, wgan_critic, wgan_metrics = train_wgan(
    dataloader, 
    num_epochs=NUM_EPOCHS,
    clip_value=WGAN_CLIP_VALUE,
    learning_rate=WGAN_LEARNING_RATE,
    use_wandb=USE_WANDB
)

print("\n[3/3] Entrenando WGAN-GP...")
print("-"*70)

wgangp_gen, wgangp_critic, wgangp_metrics = train_wgangp(
    dataloader, 
    num_epochs=NUM_EPOCHS,
    lambda_gp=WGANGP_LAMBDA,
    learning_rate=WGANGP_LEARNING_RATE,
    use_wandb=USE_WANDB
)

print("\n" + "="*70)
print("EXPERIMENTO 1 COMPLETADO")
print("="*70)

## Experimento 2: Comparación General - Evolución del Entrenamiento

Comparamos las curvas de entrenamiento y muestras generadas de las tres variantes para visualizar:
1. **Vanilla GAN**: Pérdidas no correlacionadas con calidad, posible inestabilidad
2. **WGAN**: Wasserstein distance como métrica significativa, pero con limitaciones del clipping
3. **WGAN-GP**: Mejora sobre WGAN con gradient penalty

In [ ]:
plot_training_curves_all3(vanilla_metrics, wgan_metrics, wgangp_metrics)

In [ ]:
compare_samples_all3(vanilla_metrics, wgan_metrics, wgangp_metrics)

In [ ]:
print("Evolución de muestras generadas durante el entrenamiento:\n")

plot_sample_evolution(vanilla_metrics, "Vanilla GAN")
plot_sample_evolution(wgan_metrics, "WGAN")
plot_sample_evolution(wgangp_metrics, "WGAN-GP")

## Experimento 3: Análisis de Estabilidad de Gradientes

Uno de los problemas clave de Vanilla GAN es el **vanishing gradient problem**: cuando el discriminador es muy bueno, los gradientes que recibe el generador se vuelven muy pequeños.

WGAN soluciona esto parcialmente con Wasserstein distance, pero el weight clipping puede causar gradientes erráticos.

WGAN-GP debería mostrar los gradientes más estables y consistentes.

In [ ]:
plot_gradient_norms_all3(vanilla_metrics, wgan_metrics, wgangp_metrics)

## Experimento 4: Análisis de Distribución de Pesos (WGAN vs WGAN-GP)

Un problema fundamental del weight clipping en WGAN es que **fuerza los pesos hacia los límites** [-c, c], reduciendo artificialmente la capacidad del modelo.

WGAN-GP no tiene esta restricción y debería mostrar una distribución de pesos más natural y amplia.

**Nota**: Solo comparamos WGAN vs WGAN-GP aquí porque Vanilla GAN no tiene este problema específico del clipping.

In [ ]:
plot_weight_distributions(wgan_metrics, wgangp_metrics)

## Experimento 5: Sensibilidad a Hiperparámetros

**Objetivo**: Demostrar empíricamente que WGAN-GP es más robusta a variaciones en sus hiperparámetros que WGAN.

Evaluamos cómo cada método responde a diferentes valores de sus hiperparámetros principales:
- **WGAN**: c ∈ {0.001, 0.01, 0.1}
- **WGAN-GP**: λ ∈ {1, 10, 100}

**Hipótesis**: WGAN debería ser muy sensible al valor de c (muy pequeño causa vanishing gradients, muy grande viola la restricción de Lipschitz), mientras que WGAN-GP debería ser más robusto a diferentes valores de λ.


In [ ]:
print("EXPERIMENTO 5: Sensibilidad a Hiperparámetros")
print(f"Épocas por experimento: {NUM_EPOCHS}\n")

wgan_sensitivity_results = {
    0.01: wgan_metrics
}

for clip_val in [0.001, 0.1]:
    print(f"WGAN con c={clip_val}")
    _, _, metrics = train_wgan(
        dataloader,
        num_epochs=NUM_EPOCHS,
        clip_value=clip_val,
        log_interval=200,
        use_wandb=USE_WANDB
    )
    wgan_sensitivity_results[clip_val] = metrics
    print("\n---\n")

In [ ]:
wgangp_sensitivity_results = {
    10: wgangp_metrics
}

for lambda_val in [1, 100]:
    print(f"WGAN-GP con λ={lambda_val}")
    _, _, metrics = train_wgangp(
        dataloader, 
        num_epochs=NUM_EPOCHS,
        lambda_gp=lambda_val,
        log_interval=200,
        use_wandb=USE_WANDB
    )
    wgangp_sensitivity_results[lambda_val] = metrics
    print("\n---\n")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['green', 'blue', 'orange']
for idx, (clip_val, metrics) in enumerate(wgan_sensitivity_results.items()):
    axes[0, 0].plot(metrics.iterations, metrics.wasserstein_distances, 
                    label=f'c={clip_val}', alpha=0.7, color=colors[idx])
axes[0, 0].set_title('WGAN: W-distance vs Clip Value', fontsize=12)
axes[0, 0].set_xlabel('Iteración')
axes[0, 0].set_ylabel('W-distance')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

for idx, (lambda_val, metrics) in enumerate(wgangp_sensitivity_results.items()):
    axes[0, 1].plot(metrics.iterations, metrics.wasserstein_distances, 
                    label=f'λ={lambda_val}', alpha=0.7, color=colors[idx])
axes[0, 1].set_title('WGAN-GP: W-distance vs Lambda', fontsize=12)
axes[0, 1].set_xlabel('Iteración')
axes[0, 1].set_ylabel('W-distance')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

wgan_final_samples = []
for clip_val, metrics in wgan_sensitivity_results.items():
    wgan_final_samples.append(metrics.generated_samples[-1][:4])
wgan_grid = torch.cat(wgan_final_samples, dim=0)
wgan_display = vutils.make_grid(wgan_grid, padding=2, normalize=True, nrow=4)
axes[1, 0].imshow(np.transpose(wgan_display.numpy(), (1, 2, 0)))
axes[1, 0].set_title('WGAN: Muestras (c=0.001, 0.01, 0.1)', fontsize=10)
axes[1, 0].axis('off')

wgangp_final_samples = []
for lambda_val, metrics in wgangp_sensitivity_results.items():
    wgangp_final_samples.append(metrics.generated_samples[-1][:4])
wgangp_grid = torch.cat(wgangp_final_samples, dim=0)
wgangp_display = vutils.make_grid(wgangp_grid, padding=2, normalize=True, nrow=4)
axes[1, 1].imshow(np.transpose(wgangp_display.numpy(), (1, 2, 0)))
axes[1, 1].set_title('WGAN-GP: Muestras (λ=1, 10, 100)', fontsize=10)
axes[1, 1].axis('off')

plt.suptitle('Experimento 5: Sensibilidad a Hiperparámetros', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/hyperparameter_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

---

# Análisis y Conclusiones

---

## Resumen de Resultados

In [ ]:
# Empty

## Conclusiones Esperadas Teóricamente

Basándose en los papers de Vanilla GAN, WGAN y WGAN-GP, se esperan los siguientes resultados:

### Hallazgos Esperados por Arquitectura

#### Vanilla GAN (Goodfellow et al., 2014)
**Problemas esperados:**
1. **Pérdidas no correlacionadas**: BCE loss no indica calidad de generación. Las pérdidas pueden oscilar erráticamente incluso cuando las muestras mejoran.
2. **Vanishing gradients**: Cuando D es muy bueno (cercano a 1 para reales y 0 para fakes), los gradientes para G se vuelven muy pequeños.
3. **Mode collapse**: El generador puede colapsar a generar pocas variantes de muestras.
4. **Training instability**: El balance entre D y G es delicado y puede desestabilizarse.

#### WGAN (Arjovsky et al., 2017)
**Mejoras sobre Vanilla GAN:**
1. **Wasserstein distance como métrica correlacionada**: La pérdida indica directamente qué tan bueno es el modelo.
2. **No vanishing gradients**: Incluso con un Critic óptimo, el generador recibe gradientes útiles.
3. **Mayor estabilidad**: El entrenamiento es más robusto.

**Problemas introducidos:**
1. **Weight clipping es una restricción tosca**: Fuerza los pesos hacia los extremos [-c, c].
2. **Capacidad reducida**: Los pesos restringidos limitan la expresividad del modelo.
3. **Gradientes patológicos**: El clipping puede causar gradientes que explotan o desaparecen.

#### WGAN-GP (Gulrajani et al., 2017)
**Mejoras sobre WGAN:**
1. **Gradient penalty en vez de clipping**: Restricción suave que no limita artificialmente los pesos.
2. **Distribución de pesos natural**: Los pesos pueden tomar cualquier valor, maximizando la capacidad del modelo.
3. **Gradientes más estables**: La penalización suave evita los problemas del clipping abrupto.
4. **Menor sensibilidad a hiperparámetros**: Más robusto a variaciones en λ comparado con c en WGAN.

### Validación Esperada de las Afirmaciones de los Papers

Los experimentos deberían confirmar:

1. **Vanilla GAN → WGAN**: 
   - La W-distance en WGAN debería correlacionar con calidad de generación
   - WGAN debería ser más estable que Vanilla GAN
   - Los gradientes en WGAN no deberían desaparecer como en Vanilla GAN

2. **WGAN → WGAN-GP**:
   - Gradient penalty es superior a weight clipping para enforcer la restricción de Lipschitz
   - Mayor estabilidad de entrenamiento con gradient penalty
   - Mejor utilización de la capacidad del modelo (pesos no restringidos)
   - Menor sensibilidad a hiperparámetros

---

## Mis Observaciones y Análisis Basados en los Resultados Obtenidos

[A COMPLETAR]

## Guardar Modelos Entrenados

In [ ]:
torch.save(vanilla_gen.state_dict(), f'{MODELS_DIR}/vanilla_generator.pth')
torch.save(vanilla_disc.state_dict(), f'{MODELS_DIR}/vanilla_discriminator.pth')
torch.save(wgan_gen.state_dict(), f'{MODELS_DIR}/wgan_generator.pth')
torch.save(wgan_critic.state_dict(), f'{MODELS_DIR}/wgan_critic.pth')
torch.save(wgangp_gen.state_dict(), f'{MODELS_DIR}/wgangp_generator.pth')
torch.save(wgangp_critic.state_dict(), f'{MODELS_DIR}/wgangp_critic.pth')

print("Modelos guardados exitosamente:")
print(f"  ✓ Vanilla GAN (Generador + Discriminador)")
print(f"  ✓ WGAN (Generador + Critic)")
print(f"  ✓ WGAN-GP (Generador + Critic)")
print(f"\nUbicación: {MODELS_DIR}/")

---
## Referencias

1. **Goodfellow, I., Pouget-Abadie, J., Mirza, M., Xu, B., Warde-Farley, D., Ozair, S., Courville, A., & Bengio, Y. (2014).** Generative Adversarial Nets. *Advances in Neural Information Processing Systems (NeurIPS)*.  
   [https://arxiv.org/abs/1406.2661](https://arxiv.org/abs/1406.2661)

2. **Arjovsky, M., Chintala, S., & Bottou, L. (2017).** Wasserstein GAN.  
   [arXiv:1701.07875](https://arxiv.org/abs/1701.07875)

3. **Gulrajani, I., Ahmed, F., Arjovsky, M., Dumoulin, V., & Courville, A. (2017).** Improved Training of Wasserstein GANs.  
   [arXiv:1704.00028](https://arxiv.org/abs/1704.00028)

---
